In [1]:
# Load all the environment variables from ~/.bashrc
%load_ext autoreload
%autoreload 2
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

paths = []
if os.path.join(os.path.expanduser("~"), ".bashrc"):
    paths.append(os.path.join(os.path.expanduser("~"), ".bashrc"))
if os.path.join(os.path.expanduser("~"), ".zshrc"):
    paths.append(os.path.join(os.path.expanduser("~"), ".zshrc"))

for path in paths:
    with open(path) as f:
        lines = f.readlines()
        for line in lines:
            if line.startswith("export"):
                var = line.split()[1].split("=")[0]
                if var == "PATH":
                    continue
                os.environ[var] = line.split("=")[1].strip().strip('"')
                print(var, os.environ[var])

OPENSSL_ROOT_DIR /usr/local/opt/openssl@3
ZSH /Users/guillemcv/.oh-my-zsh
PL_API_KEY PLAKfda103372c864062b8900aaa4c9639bd
LUPNT_DATA_PATH /Users/guillemcv/Development/NavLab/LuPNT/data
CLOUDSDK_PYTHON $(which python)
CODECOV_TOKEN d0f7e7f7-f53c-4a81-8d30-2539383403f5
GMAT_PATH /Users/guillemcv/Applications/GMAT R2022a
PYTHONPATH ${PYTHONPATH}:/Users/guillemcv/Development/NavLab/LuPNT/


In [2]:
import pylupnt as pnt
import numpy as np
import os
from utils.gmat import gmat, gmat_path
from utils import data

In [3]:
kwargs = {}
coe = data.coe_array_elfo
earth_potential_file_path = os.path.join(
    gmat_path, "data", "gravity", "earth", "JGM3.cof"
)
luna_potential_file_path = os.path.join(
    gmat_path, "data", "gravity", "luna", "grgm900c.cof"
)

EarthMJ2000Eq = gmat.Construct(
    "CoordinateSystem", "EarthMJ2000Eq", "Earth", "MJ2000Eq")
LunaMJ2000Eq = gmat.Construct(
    "CoordinateSystem", "LunaMJ2000Eq", "Luna", "MJ2000Eq")

# Spacecraft configuration preliminaries
sc = gmat.Construct("Spacecraft", "LunaOrbiter")
sc.SetField("DateFormat", "UTCGregorian")
sc.SetField("Epoch", "20 Jul 2020 12:00:00.000")
sc.SetField("CoordinateSystem", "LunaMJ2000Eq")
# sc.SetField("DisplayStateType", "Keplerian")

# Orbital state
sc.SetField("SMA", coe[0])
sc.SetField("ECC", coe[1])
sc.SetField("INC", np.rad2deg(coe[2]))
sc.SetField("RAAN", np.rad2deg(coe[3]))
sc.SetField("AOP", np.rad2deg(coe[4]))
sc.SetField("TA", np.rad2deg(pnt.mean_to_true_anomaly(coe[5], coe[1])))

# Spacecraft ballistic properties for the SRP and Drag models
if "SRPArea" in kwargs:
    sc.SetField("SRPArea", 2.5)
if "Cr" in kwargs:
    sc.SetField("Cr", 1.75)
if "DragArea" in kwargs:
    sc.SetField("DragArea", 1.8)
if "Cd" in kwargs:
    sc.SetField("Cd", 2.1)

sc.SetField("DryMass", 80)

# Force model settings
fm = gmat.Construct("ForceModel", "FM")
fm.SetField("CentralBody", "Luna")

# An 8x8 JGM-3 Gravity Model
grav = gmat.Construct("GravityField")
grav.SetField("BodyName", "Luna")
grav.SetField("PotentialFile", luna_potential_file_path)
grav.SetField("Degree", 0)
grav.SetField("Order", 0)

# Add forces into the ODEModel container
fm.AddForce(grav)

gmat.Initialize()

# Build the propagation container class
pdprop = gmat.Construct("Propagator", "PDProp")

# Create and assign a numerical integrator for use in the propagation
gator = gmat.Construct("PrinceDormand78", "Gator")
pdprop.SetReference(gator)

# Set some of the fields for the integration
pdprop.SetField("InitialStepSize", 60.0)
pdprop.SetField("Accuracy", 1.0e-12)
pdprop.SetField("MinStep", 0.0)

# Perform top level initialization
gmat.Initialize()

# Setup the spacecraft that is propagated
pdprop.AddPropObject(sc)
pdprop.PrepareInternals()

# Refresh the 'gator reference
fm = pdprop.GetODEModel()
gator = pdprop.GetPropagator()

# Take a 60 second step, showing the state before and after propagation
print(sc.GetEpoch())
print(gator.GetState())
print(sc.GetState().GetState())
print(sc.GetCartesianState())

29051.000428638676
[-162550.00497932723, 305071.7865004737, 155929.03244103043, -1.0433590712049405, -1.3773987417408444, 0.12840375330478881]
[-162550.00497932723, 305071.7865004737, 155929.03244103043, -1.0433590712049405, -1.3773987417408444, 0.12840375330478881]
-2857.618683217181 -2997.994311239047 5731.496383057383 -0.1222975129350246 -0.8992864018240367 0.2452908630394531


In [5]:
EarthMJ2000Eq = gmat.Construct(
    "CoordinateSystem", "EarthMJ2000Eq", "Earth", "MJ2000Eq")
LunaMJ2000Eq = gmat.Construct(
    "CoordinateSystem", "LunaMJ2000Eq", "Luna", "MJ2000Eq")
epoch = gmat.GmatTime(sc.GetEpoch())
state_earth = gmat.Rvector6(*gator.GetState())
state_luna = gmat.Rvector6()
gmat.Initialize()

In [6]:
converter = gmat.CoordinateConverter()
converter.Convert(epoch, state_earth, EarthMJ2000Eq, state_luna, LunaMJ2000Eq)
print(state_luna)

-2857.618683105946 -2997.994311181305 5731.496383071499 -0.1222975129351668 -0.8992864018237618 0.2452908630395873


In [10]:
state_luna.GetSize()

6

In [7]:
cart = pnt.classical_to_cartesian(data.coe_array_elfo, pnt.MU_MOON)
epoch = pnt.SpiceInterface.string_to_tai("2001/04/06 12:00:00.000 UTC")
pnt.CoordConverter.convert(
    epoch, cart, pnt.CoordSystem.MI, pnt.CoordSystem.GCRF)

array([ 3.59752007e+05, -2.25888205e+04, -3.62560271e+04, -5.81745120e-03,
        9.39098791e-02,  6.46562347e-01])

In [43]:
time_sys_converter = gmat.TimeSystemConverter.Instance()
GMAT_CSPICE_TAI_OFFSET = time_sys_converter.ConvertGregorianToMjd("01 Jan 2000 12:00:00.000") * 86400.0
print(GMAT_CSPICE_TAI_OFFSET)

1861488000.0


In [31]:
time_sys_converter = gmat.TimeSystemConverter.Instance()
epoch_gmat = (
    time_sys_converter.Convert(
        # time_sys_converter.ConvertGregorianToMjd("20 Jul 2020 12:00:00.000"),
        time_sys_converter.ConvertGregorianToMjd("20 Jul 2020 12:00:00.000"),
        gmat.TimeSystemConverter.UTC,
        gmat.TimeSystemConverter.TAI,
    )
)
print(epoch_gmat)

648518437.0343814


In [27]:
epoch = pnt.SpiceInterface.string_to_tai("2020/07/20 12:00:00.000 UTC")
print(epoch)

648518437.0


In [28]:
gmat_helpers.convert_pylupnt_to_gmat_epoch(epoch)

29051.00042824074

In [29]:
epoch_gmat = time_sys_converter.ConvertGregorianToMjd("20 Jul 2020 12:00:00.000")
print(epoch_gmat)

29051.0


In [30]:
gmat_helpers.convert_gmat_to_pylupnt_epoch(epoch_gmat)

648518400.0

In [12]:
cart_MI = pnt.classical_to_cartesian(data.coe_array_elfo, pnt.MU_MOON)
epoch = pnt.SpiceInterface.string_to_tai("2020/07/20 12:00:00.000")

# coord_systems = pnt.CoordSystem.__members__.values()
coord_systems = [pnt.ITRF, pnt.GCRF, pnt.ICRF, pnt.MI, pnt.EMR, pnt.ME, pnt.PA]

In [7]:
from utils import gmat_helpers


In [8]:
cart_from = pnt.CoordConverter.convert(
    epoch, cart_MI, pnt.MI, pnt.GCRF
)
print(cart_from)

[ 1.56834780e+05 -3.11067770e+05 -1.44466037e+05  7.98764039e-01
 -4.21173985e-01  3.62177976e-01]


In [13]:
cart_from_gmat = gmat_helpers.convert_coord(
    epoch, cart_MI, pnt.MI, pnt.GCRF
)
print(cart_from_gmat)

APIException: PlanetaryEphem (sub)class exception: Requested epoch 7506.250163762 is not on the DE file '/Users/guillemcv/Applications/GMAT R2022a/data/planetary_ephem/de/leDE1941.405'.


In [14]:
time_sys_converter.ConvertGregorianToMjd("20 Jul 2020 12:00:00.000")

29051.0